In [92]:
# 2. Cleaned data → data/processed/
# This is the cleaned version after fixing:

# dates

# amounts

# categories

# duplicates


In [93]:
import pandas as pd
import boto3

path = "s3://ds-learning-hunny/aws-practice-data/banking/data/raw-data/fake_bank_transactions.csv"

df = pd.read_csv(path)
df.head()

,account_name,bank_name,month,date,transaction_type,description,category,amount,balance
0,Maya Johnson,Practice Data Bank,2026-01,2026-01-02,deposit,Paycheck,Income,2450.00,2450.00
1,Maya Johnson,Practice Data Bank,2026-01,2026-01-03,withdrawal,Rent Payment,Housing,-1200.00,1250.00
2,Maya Johnson,Practice Data Bank,2026-01,2026-01-05,withdrawal,Duke Energy Bill,Utilities,-86.45,1163.55
3,Maya Johnson,Practice Data Bank,2026-01,2026-01-08,withdrawal,Walmart Groceries,Groceries,-74.22,1089.33
4,Maya Johnson,Practice Data Bank,2026-01,2026-01-12,withdrawal,Netflix Subscription,Entertainment,-15.49,1073.84


In [94]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   account_name      21 non-null     object 
 1   bank_name         21 non-null     object 
 2   month             21 non-null     object 
 3   date              21 non-null     object 
 4   transaction_type  21 non-null     object 
 5   description       21 non-null     object 
 6   category          21 non-null     object 
 7   amount            21 non-null     float64
 8   balance           21 non-null     float64
dtypes: float64(2), object(7)
memory usage: 1.6+ KB


In [95]:
# 1. convert date column
df['date'] = pd.to_datetime(df['date'])
df['date'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 21 entries, 0 to 20
Series name: date
Non-Null Count  Dtype         
--------------  -----         
21 non-null     datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 300.0 bytes


In [96]:
# 2 check for duplicates
df.drop_duplicates()

,account_name,bank_name,month,date,transaction_type,description,category,amount,balance
0,Maya Johnson,Practice Data Bank,2026-01,2026-01-02,deposit,Paycheck,Income,2450.00,2450.00
1,Maya Johnson,Practice Data Bank,2026-01,2026-01-03,withdrawal,Rent Payment,Housing,-1200.00,1250.00
2,Maya Johnson,Practice Data Bank,2026-01,2026-01-05,withdrawal,Duke Energy Bill,Utilities,-86.45,1163.55
3,Maya Johnson,Practice Data Bank,2026-01,2026-01-08,withdrawal,Walmart Groceries,Groceries,-74.22,1089.33
4,Maya Johnson,Practice Data Bank,2026-01,2026-01-12,withdrawal,Netflix Subscription,Entertainment,-15.49,1073.84
5,Maya Johnson,Practice Data Bank,2026-01,2026-01-18,withdrawal,Shell Gas,Transportation,-42.10,1031.74
6,Maya Johnson,Practice Data Bank,2026-01,2026-01-25,withdrawal,Phone Bill,Utilities,-68.00,963.74
7,Maya Johnson,Practice Data Bank,2026-02,2026-02-01,deposit,Paycheck,Income,2450.00,3413.74
8,Maya Johnson,Practice Data Bank,2026-02,2026-02-03,withdrawal,Rent Payment,Housing,-1200.00,2213.74
9,Maya Johnson,Practice Data Bank,2026-02,2026-02-06,withdrawal,Duke Energy Bill,Utilities,-91.30,2122.44


In [97]:
# 3 check for missing values
df.isnull().sum()

df = df.dropna(subset=[
    'date',
    'amount',
    'balance',
    'transaction_type',
    'description',
    'category'
    ])
# These are optional metadata fields:
# account_name
# bank_name
# month

df = df.drop(columns=['account_name','bank_name'])

# make sures that moenth columns are filled 
df['month']= df['date'].dt.strftime('%Y-%m')
df.head()

,month,date,transaction_type,description,category,amount,balance
0,2026-01,2026-01-02,deposit,Paycheck,Income,2450.00,2450.00
1,2026-01,2026-01-03,withdrawal,Rent Payment,Housing,-1200.00,1250.00
2,2026-01,2026-01-05,withdrawal,Duke Energy Bill,Utilities,-86.45,1163.55
3,2026-01,2026-01-08,withdrawal,Walmart Groceries,Groceries,-74.22,1089.33
4,2026-01,2026-01-12,withdrawal,Netflix Subscription,Entertainment,-15.49,1073.84


In [98]:
# 4 Ensure amount and balance is numeric
df['amount'] = pd.to_numeric(df['amount'])
df['balance'] = pd.to_numeric(df['balance'])


In [99]:
# 5 Validate balance
validation_df = df[['amount','balance']]
validation_df['expected_balance'] = (validation_df['balance'].shift(1)+ validation_df['amount'])
validation_df.loc[0,'expected_balance'] = validation_df.loc[0,'balance']
validation_df.head()

C:\Users\hunny\AppData\Local\Temp\ipykernel_17164\4074619601.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df['expected_balance'] = (validation_df['balance'].shift(1)+ validation_df['amount'])


,amount,balance,expected_balance
0,2450.00,2450.00,2450.00
1,-1200.00,1250.00,1250.00
2,-86.45,1163.55,1163.55
3,-74.22,1089.33,1089.33
4,-15.49,1073.84,1073.84


In [100]:
# NOW VALIDATE
validation_df['is_valid'] = (validation_df['balance'] - validation_df['expected_balance']).abs()<0.01

print("Not Valid Balance") if (validation_df['is_valid'] == False).any() else print("Valid Balance!")


Valid Balance!


C:\Users\hunny\AppData\Local\Temp\ipykernel_17164\1004980842.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df['is_valid'] = (validation_df['balance'] - validation_df['expected_balance']).abs()<0.01


In [101]:
df.reset_index()

,index,month,date,transaction_type,description,category,amount,balance
0,0,2026-01,2026-01-02,deposit,Paycheck,Income,2450.00,2450.00
1,1,2026-01,2026-01-03,withdrawal,Rent Payment,Housing,-1200.00,1250.00
2,2,2026-01,2026-01-05,withdrawal,Duke Energy Bill,Utilities,-86.45,1163.55
3,3,2026-01,2026-01-08,withdrawal,Walmart Groceries,Groceries,-74.22,1089.33
4,4,2026-01,2026-01-12,withdrawal,Netflix Subscription,Entertainment,-15.49,1073.84
5,5,2026-01,2026-01-18,withdrawal,Shell Gas,Transportation,-42.10,1031.74
6,6,2026-01,2026-01-25,withdrawal,Phone Bill,Utilities,-68.00,963.74
7,7,2026-02,2026-02-01,deposit,Paycheck,Income,2450.00,3413.74
8,8,2026-02,2026-02-03,withdrawal,Rent Payment,Housing,-1200.00,2213.74
9,9,2026-02,2026-02-06,withdrawal,Duke Energy Bill,Utilities,-91.30,2122.44


In [102]:
df.to_csv('s3://ds-learning-hunny/aws-practice-data/banking/data/processed/bank_transactions_cleaned.csv',index=False)

In [107]:
df.to_csv('C:/Users/hunny/OneDrive/Documents/Resume/JOBSTUDY/Data_Science_Projects/AWS_PROJECT/banking-project/data/processed/bank_transactions_cleaned.csv', index=False)
